# 01 · Exploratory data analysis: SIM interventions, 2020 – 2026-09-24

**Purpose.** Investigate the two SIM intervention files in `data/raw/` following the course
*Dataset Investigation Guide* (its ten steps are the section headings below) and standard EDA
practice, **in service of the team-accepted v2 framing**:

> Forecast, one calendar day ahead, the number of SIM interventions **published in the open data**
> for each operational Division 1–6. Forecast for day *D* is prepared on *D−1* with completed
> daily counts through *D−3*. Baseline: recent-window mean of the same weekday (window chosen on
> validation data). Primary metric: MAE (interventions/day), plus bias. Volume ≠ workload.

Sources of the framing: `Business_Problem_Framing_Team4_DEFINITIVE.docx`,
`Dataset_Investigation_Guide_Firefighter_Team4.docx` and `verification/09_FINAL_VERIFIED.md`
(all v2). Every number the v2 documents state is **recomputed here** from the local files and
checked; a mismatch stops the notebook.

**Leakage discipline.** Steps 1–9 describe the whole 2020 – 2026 period, because drift is a data
finding. Anything that could inform modelling choices (seasonality, autocorrelation, dispersion,
baseline windows) is estimated on **2021 – 2024 only**, the proposed training/validation window.
2025 → 2026-09-24 is the proposed test period and is only inspected for level changes.

Run: open in VS Code with the `.venv` kernel, or
`uv run jupyter nbconvert --to notebook --execute notebooks/01_eda.ipynb --output-dir /tmp`.
Figures go to `reports/figures/`, key numbers to `reports/eda_numbers.json`, and the
division × day table to `data/processed/division_day.parquet`.

In [ ]:
import json
import os
import warnings
from datetime import date, datetime
from itertools import pairwise
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
from cycler import cycler
from matplotlib.figure import Figure
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tools.sm_exceptions import InterpolationWarning
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.stattools import acf, adfuller, kpss

from ycit440_sim_forecast.data import (
    RAW_FILES,
    TARGET_DIVISIONS,
    daily_counts,
    load_interventions,
)
from ycit440_sim_forecast.manifest import build_manifest

# VS Code starts notebooks in notebooks/; every path below is relative to the repo root.
if not Path("pyproject.toml").exists():
    os.chdir(Path.cwd().parent)

RAW = Path("data/raw")
FIG = Path("reports/figures")
FIG.mkdir(parents=True, exist_ok=True)
Path("data/processed").mkdir(parents=True, exist_ok=True)

RETRIEVED = date(2026, 9, 26)  # v2 Dataset Investigation Guide, step 1
TRAIN = (date(2021, 1, 1), date(2024, 12, 30))  # structure is estimated here only
R: dict[str, object] = {}  # key numbers, written to reports/eda_numbers.json

pl.Config.set_tbl_rows(30)
pl.Config.set_tbl_cols(20)
pl.Config.set_fmt_str_lengths(60)


def check(cond: bool, msg: str) -> None:
    """Stop the notebook when a documented v2 number does not reproduce."""
    if not cond:
        raise AssertionError(msg)


# Reference palette: categorical slots in fixed order, one per division.
SERIES = [
    "#2a78d6",
    "#eb6834",
    "#1baf7a",
    "#eda100",
    "#e87ba4",
    "#008300",
    "#4a3aa7",
    "#e34948",
]
DIV_COLOR = dict(zip(TARGET_DIVISIONS, SERIES, strict=False))
INK, INK2, MUTED, GRID, AXIS = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7"
plt.rcParams.update(
    {
        "figure.facecolor": "#fcfcfb",
        "axes.facecolor": "#fcfcfb",
        "axes.edgecolor": AXIS,
        "axes.labelcolor": INK2,
        "axes.titlecolor": INK,
        "axes.titlesize": 11,
        "axes.titleweight": "bold",
        "axes.titlelocation": "left",
        "axes.grid": True,
        "axes.axisbelow": True,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.prop_cycle": cycler(color=SERIES),
        "grid.color": GRID,
        "grid.linewidth": 0.6,
        "xtick.color": MUTED,
        "ytick.color": MUTED,
        "xtick.labelcolor": INK2,
        "ytick.labelcolor": INK2,
        "legend.frameon": False,
        "legend.labelcolor": INK2,
        "lines.linewidth": 1.5,
        "font.family": "sans-serif",
        "font.size": 9,
        "figure.dpi": 110,
    }
)


def save(fig: Figure, name: str) -> None:
    fig.savefig(FIG / f"{name}.png", dpi=150, bbox_inches="tight")

## Load

`load_interventions` reads both files with an explicit schema (no inference), turns empty
strings into nulls, parses the 8 date-only timestamps to midnight with `has_time = False`, and
records the `source_file`. It lives in `src/` and is unit-tested.

In [ ]:
df = load_interventions(RAW)
SRC_COLS = [
    c for c in df.columns if c not in {"has_time", "date", "year", "source_file"}
]
check(df.height == 754_413, f"rows {df.height} != 754,413 (v2)")
df.head()

## Step 1 · Source and version

| Field | Value |
|---|---|
| Dataset | *Interventions des pompiers de Montréal* |
| Publisher | Ville de Montréal, Service de sécurité incendie de Montréal (SIM) |
| Source | https://donnees.montreal.ca/dataset/interventions-service-securite-incendie-montreal |
| Source system | SIM computer-assisted dispatch (RAO); extracted for provincial reporting |
| Files used | *Interventions du SIM – 2020 à 2024* and *– courant (2 dernières années)*, CSV |
| Retrieved | 2026-09-26 |
| Update frequency | daily (official coverage 2005-01-01 → present) |
| Licence | CC BY 4.0 |

The table below pins the exact bytes (sha256, also committed in `data/manifest.json`).

In [ ]:
manifest = build_manifest(RAW)
files = (
    df.group_by("source_file")
    .agg(
        rows=pl.len(),
        first=pl.col("CREATION_DATE_TIME").min(),
        last=pl.col("CREATION_DATE_TIME").max(),
    )
    .with_columns(
        bytes=pl.col("source_file").map_elements(
            lambda f: manifest[f"{RAW.as_posix()}/{f}"]["bytes"], return_dtype=pl.Int64
        ),
        sha256_12=pl.col("source_file").map_elements(
            lambda f: str(manifest[f"{RAW.as_posix()}/{f}"]["sha256"])[:12],
            return_dtype=pl.String,
        ),
    )
    .sort("first")
)
check(
    files["rows"].to_list() == [539_203, 215_210], "per-file row counts differ from v2"
)
R["files"] = files.with_columns(pl.col("first", "last").dt.to_string()).to_dicts()
files

## Step 2 · What one row represents

One row = one intervention **event** recorded in SIM's dispatch system, with its *final*
classification as extracted. `INCIDENT_NBR` restarts every year, so the candidate key is
`(year, INCIDENT_NBR)`. We test that, then test **completeness**: if numbers are assigned
sequentially, the gap between the highest number and the published rows is the share of
dispatch events that never reach the open data.

In [ ]:
dup_key = df.height - df.select("year", "INCIDENT_NBR").n_unique()
dup_nbr_only = df.height - df["INCIDENT_NBR"].n_unique()
dup_rows = df.height - df.select(SRC_COLS).n_unique()
check(dup_key == 0, "(year, INCIDENT_NBR) is not unique")
R["duplicates"] = {"year_nbr": dup_key, "nbr_only": dup_nbr_only, "full_rows": dup_rows}
print(f"duplicate (year, INCIDENT_NBR): {dup_key}")
print(f"duplicate INCIDENT_NBR alone:   {dup_nbr_only:,} (restarts yearly)")
print(f"fully duplicated rows:          {dup_rows}")

# Totals from SIM's annual activity reports (v2 red-team finding H).
SIM_REPORTED = {2022: 118_916, 2023: 128_902, 2024: 129_822}
published = (
    df.with_columns(main=pl.col("INCIDENT_NBR") < 500_000)
    .group_by("year")
    .agg(
        published=pl.len(),
        max_nbr_main=pl.col("INCIDENT_NBR").filter(pl.col("main")).max(),
        high_range=(~pl.col("main")).sum(),
    )
    .sort("year")
    .with_columns(
        share_of_max_nbr=pl.col("published") / pl.col("max_nbr_main"),
        sim_reported=pl.col("year").replace_strict(SIM_REPORTED, default=None),
    )
    .with_columns(share_of_sim_total=pl.col("published") / pl.col("sim_reported"))
)
R["published_share"] = published.to_dicts()
published

**Reading.** The key is unique; the incident number alone is not. In every full year the open
data hold about 92–94 % of the numbers issued, and for 2022–2024 the published counts are
≈ 94 % of the totals SIM reports. The target is therefore *interventions published in the open
data*, not all SIM interventions; why ~6 % are unpublished is not documented. Since 2022 a second
numbering range (≥ 500 000) carries a handful of rows per year.

## Step 3 · Entities and relationships

Entities: the **event**; its **type** (`INCIDENT_TYPE_DESC`) and **group**
(`DESCRIPTION_GROUPE`); two **territorial labels**, `CASERNE` (station responsible for the
territory) and `DIVISION` (division responsible for the territory); municipality/borough; a
displaced location; and a vehicle count. The **responding** units, crews and times are absent,
so CASERNE/DIVISION describe *where demand arose*, not *who did the work*.

Does CASERNE nest in DIVISION? If it did, division counts could be rebuilt from casernes.

In [ ]:
cd = df.filter(pl.col("CASERNE") > 0).group_by("CASERNE", "DIVISION").len()
per_caserne = (
    cd.group_by("CASERNE")
    .agg(
        n_divisions=pl.len(),
        n_divisions_1_6=pl.col("DIVISION").is_in(TARGET_DIVISIONS).sum(),
        rows=pl.col("len").sum(),
        modal_rows=pl.col("len").max(),
    )
    .with_columns(modal_share=pl.col("modal_rows") / pl.col("rows"))
    .sort("modal_share")
)
nest = {
    "casernes": per_caserne.height,
    "casernes_in_several_divisions": int((per_caserne["n_divisions"] > 1).sum()),
    "casernes_in_several_divisions_1_6": int(
        (per_caserne["n_divisions_1_6"] > 1).sum()
    ),
    "rows_in_modal_division": per_caserne.select(
        pl.col("modal_rows").sum() / pl.col("rows").sum()
    ).item(),
}
R["caserne_nesting"] = nest
print(nest)
per_caserne.head(8)

In [ ]:
div0 = df.filter(pl.col("DIVISION") == 0)
check(div0.height == 600, f"DIVISION=0 rows {div0.height} != 600 (v2)")
check(df.filter(pl.col("CASERNE") == 0).height == 1, "CASERNE=0 rows != 1 (v2)")
R["division_0"] = {
    "rows": div0.height,
    "by_year": div0.group_by("year").len().sort("year").to_dicts(),
    "top_groups": div0["DESCRIPTION_GROUPE"].value_counts(sort=True).head(5).to_dicts(),
}
print("DIVISION = 0 by year:", div0.group_by("year").len().sort("year").rows())
print(div0["NOM_VILLE"].value_counts(sort=True).head(5))
div0["DESCRIPTION_GROUPE"].value_counts(sort=True)

**Reading.** 94.6 % of rows sit in their caserne's modal division, but a large share of casernes
report into more than one division, so CASERNE → DIVISION is **not a hierarchy**: DIVISION is
used as published, never rebuilt from casernes. `DIVISION = 0` (600 rows) is undocumented,
spread across years and groups; it is excluded from the target and reported as such.

## Step 4 · Variable reference (profile)

In [ ]:
profile = pl.DataFrame(
    [
        {
            "variable": c,
            "dtype": str(df[c].dtype),
            "nulls": df[c].null_count(),
            "null_pct": round(100 * df[c].null_count() / df.height, 3),
            "n_unique": df[c].n_unique(),
            "min": str(df[c].min())
            if df[c].dtype.is_numeric() or df[c].dtype.is_temporal()
            else "",
            "max": str(df[c].max())
            if df[c].dtype.is_numeric() or df[c].dtype.is_temporal()
            else "",
            "example": str(df[c].drop_nulls()[0]),
        }
        for c in SRC_COLS
    ]
)
nulls = dict(zip(profile["variable"], profile["nulls"], strict=True))
check(nulls["DESCRIPTION_GROUPE"] == 1_788, "DESCRIPTION_GROUPE nulls != 1,788 (v2)")
check(nulls["NOMBRE_UNITES"] == 156, "NOMBRE_UNITES nulls != 156 (v2)")
R["nulls"] = {k: v for k, v in nulls.items() if v}
profile

In [ ]:
groups = df["DESCRIPTION_GROUPE"].fill_null("(blank)").value_counts(sort=True)
check(
    groups.filter(pl.col("DESCRIPTION_GROUPE") == "NOUVEAU")["count"].item() == 7,
    "NOUVEAU != 7 (v2)",
)
units_by_group = (
    df.with_columns(g=pl.col("DESCRIPTION_GROUPE").fill_null("(blank)"))
    .group_by("g")
    .agg(
        incidents=pl.len(),
        vehicles=pl.col("NOMBRE_UNITES").sum(),
        mean_units=pl.col("NOMBRE_UNITES").mean(),
        p99_units=pl.col("NOMBRE_UNITES").quantile(0.99),
        max_units=pl.col("NOMBRE_UNITES").max(),
    )
    .with_columns(
        pct_incidents=100 * pl.col("incidents") / pl.col("incidents").sum(),
        pct_vehicles=100 * pl.col("vehicles") / pl.col("vehicles").sum(),
    )
    .sort("incidents", descending=True)
)
R["groups"] = units_by_group.to_dicts()
units_by_group

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
ug = units_by_group.head(6)
y = np.arange(ug.height)
axes[0].barh(
    y - 0.2,
    ug["pct_incidents"],
    height=0.38,
    color=SERIES[0],
    label="share of incidents",
)
axes[0].barh(
    y + 0.2, ug["pct_vehicles"], height=0.38, color=SERIES[1], label="share of vehicles"
)
axes[0].set_yticks(y, ug["g"].to_list())
axes[0].invert_yaxis()
axes[0].set_xlabel("% of 2020 - 2026 total")
axes[0].set_title("Counts and vehicle load tell different stories")
axes[0].legend(loc="lower right")
axes[0].grid(axis="y", visible=False)

u = df["NOMBRE_UNITES"].drop_nulls().to_numpy()
axes[1].hist(
    np.clip(u, None, 40),
    bins=np.arange(0.5, 41.5, 1),
    color=SERIES[0],
    edgecolor="#fcfcfb",
    linewidth=0.5,
)
axes[1].set_yscale("log")
axes[1].set_xlabel("NOMBRE_UNITES (vehicles per incident; last bar = 40 or more)")
axes[1].set_ylabel("incidents (log)")
axes[1].set_title(f"Vehicles per incident: median {np.median(u):.0f}, max {u.max()}")
fig.tight_layout()
save(fig, "04_groups_units")

In [ ]:
top_types = df["INCIDENT_TYPE_DESC"].value_counts(sort=True).head(15)
R["top_types"] = top_types.to_dicts()
top_types

| Variable | Verified meaning | Type / unit | Availability and quality | Role in this project |
|---|---|---|---|---|
| `CREATION_DATE_TIME` | Event creation time, local civil time | datetime | Known at call creation; 8 date-only rows (still usable for daily counts) | **Builds the target** (date) and calendar features |
| `DIVISION` | Division responsible for the territory | int 0–6 | Complete; `0` = 600 undocumented rows | **Target grouping** (1–6); 0 excluded |
| `INCIDENT_NBR` | Event number, restarts yearly | int | Unique with year; ~6 % of numbers unpublished | Key / de-duplication only |
| `DESCRIPTION_GROUPE` | Broad group (6 official + `NOUVEAU`) | category | 1,788 blank (mostly fire-type), 7 `NOUVEAU` | Diagnostics only (drift by group); not in target definition |
| `INCIDENT_TYPE_DESC` | Detailed type, may be revised after the initial report | category | ~150 types; vocabulary churns yearly | Diagnostics only; future values unknown |
| `CASERNE` | Station responsible for the territory, **not** the responder | int | 1 row = 0; not nested in DIVISION | Not used (sparser grain; see step 7) |
| `NOMBRE_UNITES` | Vehicles deployed; 1 vehicle ≈ 3–5 firefighters | int, vehicles | 156 blank; heavy tail | **Not** a predictor (post-event); not the target (volume ≠ workload) |
| `NOM_VILLE`, `NOM_ARROND` | Municipality / borough (`Indéterminé` = linked city) | text | Complete | Context only |
| `LONGITUDE/LATITUDE`, `MTM8_X/Y` | Location moved to an intersection of the street segment | float | Complete; displaced | Not used |

## Step 5 · Timing and availability

**Publication lag.** The last record is the day before yesterday relative to retrieval.

In [ ]:
last_ts: datetime = df.select(pl.col("CREATION_DATE_TIME").max()).item()
lag_days = (RETRIEVED - last_ts.date()).days
R["publication"] = {
    "last_record": str(last_ts),
    "retrieved": str(RETRIEVED),
    "lag_days": lag_days,
}
print(f"last record {last_ts}, retrieved {RETRIEVED}: lag {lag_days} days")

date_only = df.filter(~pl.col("has_time")).select(
    "CREATION_DATE_TIME", "source_file", "DIVISION"
)
check(date_only.height == 8, "date-only rows != 8 (v2)")
R["date_only_rows"] = date_only.group_by("source_file").len().to_dicts()
date_only.sort("CREATION_DATE_TIME")

In [ ]:
timed = df.filter(pl.col("has_time"))
spring_forward = [
    date(2020, 3, 8),
    date(2021, 3, 14),
    date(2022, 3, 13),
    date(2023, 3, 12),
    date(2024, 3, 10),
    date(2025, 3, 9),
    date(2026, 3, 8),
]
in_gap = timed.filter(
    pl.col("date").is_in(spring_forward) & (pl.col("CREATION_DATE_TIME").dt.hour() == 2)
).height
at_midnight = timed.filter(
    pl.col("CREATION_DATE_TIME").dt.time() == datetime.min.time()
).height
R["timestamps"] = {"rows_in_dst_gap": in_gap, "rows_exactly_midnight": at_midnight}
print(f"rows in the 02:00 hour on spring-forward days: {in_gap}  -> local civil time")
print(f"timed rows exactly at 00:00:00: {at_midnight}  -> no midnight heaping")

hourly = (
    timed.filter(pl.col("date").is_between(*TRAIN))
    .group_by(pl.col("CREATION_DATE_TIME").dt.hour().alias("hour"))
    .len()
    .sort("hour")
    .with_columns(pct=100 * pl.col("len") / pl.col("len").sum())
)
fig, ax = plt.subplots(figsize=(8, 2.8))
ax.bar(hourly["hour"], hourly["pct"], color=SERIES[0], width=0.8)
ax.set_xticks(range(0, 24, 2))
ax.set_xlabel("hour of creation (local)")
ax.set_ylabel("% of interventions")
ax.set_title("Hour of day, 2021 - 2024: trough at 04:00, plateau 10:00 - 19:00")
save(fig, "05_hour_of_day")

**When each variable becomes known** (forecast for day *D*, prepared on *D−1*, data through *D−3*):

| Variable | Known when | Allowed use |
|---|---|---|
| Calendar of *D* (weekday, month, holidays) | always | predictor |
| Daily counts up to *D−3* (from `CREATION_DATE_TIME` + `DIVISION`) | ~2 days after the day ends | lagged predictor |
| Type, group, caserne, division, units **of incidents on D** | during / after *D* | never a predictor |
| `NOMBRE_UNITES` of past incidents | after the response | only as lagged aggregate, if at all |
| Revised type labels | unknown delay | treat past labels as hindsight labels |

The portal keeps no historical snapshots, so the D−3 cutoff can be *simulated* but not
replayed exactly. The observed 2-day lag supports D−3 as the design default.

## Step 6 · Identifiers and joins

The two files are **appended**, not joined. Checks: same header, no date overlap, no key
overlap. No external dataset is needed for the initial design.

In [ ]:
headers = {f: (RAW / f).open(encoding="utf-8").readline().strip() for f in RAW_FILES}
same_header = len(set(headers.values())) == 1
ranges = (
    df.group_by("source_file")
    .agg(lo=pl.col("date").min(), hi=pl.col("date").max())
    .sort("lo")
)
no_date_overlap = ranges["hi"][0] < ranges["lo"][1]
keys = [
    set(g.select("year", "INCIDENT_NBR").rows()) for _, g in df.group_by("source_file")
]
key_overlap = len(keys[0] & keys[1])
check(same_header and no_date_overlap and key_overlap == 0, "append checks failed")
R["append"] = {
    "same_header": same_header,
    "date_overlap": not no_date_overlap,
    "key_overlap": key_overlap,
}
print(R["append"])
ranges

## Step 7 · Coverage and quality

### 7a · Calendar completeness and missingness by year

In [ ]:
all_days = df.select(
    pl.date_range(pl.col("date").min(), pl.col("date").max())
).to_series()
observed = df["date"].unique()
missing_days = all_days.filter(~all_days.is_in(observed.implode())).to_list()
check(
    missing_days == [date(2024, 12, 31)],
    f"missing days {missing_days} != [2024-12-31] (v2)",
)
check(observed.len() == 2_458, f"observed days {observed.len()} != 2,458 (v2)")
R["calendar"] = {
    "days": all_days.len(),
    "observed": observed.len(),
    "missing": [str(d) for d in missing_days],
}
print(R["calendar"])

(
    df.group_by("year")
    .agg(
        rows=pl.len(),
        blank_group=pl.col("DESCRIPTION_GROUPE").is_null().sum(),
        nouveau=(pl.col("DESCRIPTION_GROUPE") == "NOUVEAU").sum(),
        blank_units=pl.col("NOMBRE_UNITES").is_null().sum(),
        date_only=(~pl.col("has_time")).sum(),
        division_0=(pl.col("DIVISION") == 0).sum(),
        n_types=pl.col("INCIDENT_TYPE_DESC").n_unique(),
    )
    .sort("year")
)

Blank groups grow over time. What are they?

In [ ]:
df.filter(pl.col("DESCRIPTION_GROUPE").is_null()).group_by("INCIDENT_TYPE_DESC").agg(
    rows=pl.len(), mean_units=pl.col("NOMBRE_UNITES").mean()
).sort("rows", descending=True).head(6)

### 7b · Classification churn

Types appearing or disappearing year to year: a model on *total* counts is immune, but any
type-level target or feature would need a harmonisation table.

In [ ]:
types_by_year = {y: set(g["INCIDENT_TYPE_DESC"]) for (y,), g in df.group_by("year")}
years = sorted(types_by_year)
churn = pl.DataFrame(
    [
        {
            "year": y,
            "types": len(types_by_year[y]),
            "new": len(types_by_year[y] - types_by_year[p]),
            "dropped": len(types_by_year[p] - types_by_year[y]),
        }
        for p, y in pairwise(years)
    ]
)
R["type_churn"] = churn.to_dicts()
churn

### 7c · Geography sanity

Coordinates are displaced to street-segment intersections. Two checks: all points fall in the
Montréal agglomeration's bounding box, and points are heavily shared (snapping).

In [ ]:
bbox = df.select(
    pl.col("LATITUDE").min().alias("lat_min"),
    pl.col("LATITUDE").max().alias("lat_max"),
    pl.col("LONGITUDE").min().alias("lon_min"),
    pl.col("LONGITUDE").max().alias("lon_max"),
)
point_use = df.group_by("LATITUDE", "LONGITUDE").len()
shared = (
    point_use.select(pl.col("len").filter(pl.col("len") > 1).sum()).item() / df.height
)
R["geo"] = {
    **bbox.to_dicts()[0],
    "distinct_points": point_use.height,
    "rows_on_shared_points": shared,
}
print(
    f"{point_use.height:,} distinct points for {df.height:,} rows; {shared:.1%} share"
)
bbox

### 7d · Density of candidate grains (why DIVISION × day)

`daily_counts` zero-fills a group with no record on a day that has records elsewhere, and
leaves days with **no records at all** (2024-12-31) as null, so a missing day is never a zero.

In [ ]:
city = daily_counts(df, by=None)
div = daily_counts(df)  # DIVISION 1-6
cas = daily_counts(df, by="CASERNE", keep=None).filter(pl.col("CASERNE") > 0)
div.write_parquet("data/processed/division_day.parquet")


def density(frame: pl.DataFrame, name: str) -> dict[str, object]:
    n = frame["n"].drop_nulls().to_numpy().astype(float)
    return {
        "grain": name,
        "cells": n.size,
        "median": float(np.median(n)),
        "mean": round(float(n.mean()), 1),
        "pct_zero": round(float(100 * (n == 0).mean()), 2),
        "pct_le_5": round(float(100 * (n <= 5).mean()), 2),
        "var_over_mean": round(float(n.var(ddof=1) / n.mean()), 1),
    }


grains = pl.DataFrame(
    [
        density(city, "citywide x day"),
        density(div, "division x day"),
        density(cas, "caserne x day"),
    ]
)
d, c = grains.row(1, named=True), grains.row(2, named=True)
check(
    d["cells"] == 14_748 and d["median"] == 53 and d["pct_zero"] == 0,
    "division-day != v2",
)
caserne_ok = c["cells"] == 162_228 and c["median"] == 4
check(caserne_ok and abs(c["pct_zero"] - 8.96) < 0.01, "caserne-day != v2")
check(
    abs(d["pct_le_5"] - 0.14) < 0.01 and abs(c["pct_le_5"] - 65.04) < 0.01,
    "<=5 shares != v2",
)
R["grains"] = grains.to_dicts()
grains

### 7e · The target series over time

Level, breaks and shocks, citywide and per division.

In [ ]:
city_s = city.with_columns(roll28=pl.col("n").rolling_mean(28, min_samples=20))
yearly = (
    city.drop_nulls()
    .group_by(pl.col("date").dt.year().alias("year"))
    .agg(mean_per_day=pl.col("n").mean().round(1), days=pl.len())
    .sort("year")
)
R["yearly_mean_per_day"] = yearly.to_dicts()

fig, ax = plt.subplots(figsize=(11, 3.6))
ax.plot(city_s["date"], city_s["n"], color=AXIS, linewidth=0.6, label="daily")
ax.plot(
    city_s["date"], city_s["roll28"], color=SERIES[0], linewidth=2, label="28-day mean"
)
for when, text in [
    (date(2020, 4, 15), "COVID: first-responder\ncalls suspended"),
    (date(2023, 4, 6), "ice storm\n1,712 on 6 Apr"),
    (date(2025, 12, 20), "late-2025\ndrop"),
]:
    ax.annotate(text, (when, 1050), fontsize=8, color=INK2, ha="center")
ax.axvspan(
    *mdates.date2num([date(2025, 1, 1), date(2026, 9, 24)]), color=GRID, alpha=0.5, lw=0
)
ax.text(
    mdates.date2num(date(2025, 1, 15)),
    150,
    "proposed test period",
    fontsize=8,
    color=MUTED,
)
ax.set_ylim(0, 1200)
ax.set_ylabel("interventions / day")
ax.set_title("Citywide published interventions per day, 2020 - 2026-09-24")
ax.legend(ncol=2, loc="upper left", bbox_to_anchor=(0, -0.08))
save(fig, "07_city_daily")
yearly

In [ ]:
div_w = (
    div.drop_nulls()
    .group_by_dynamic("date", every="1w", group_by="DIVISION")
    .agg(n=pl.col("n").mean())
    .sort("DIVISION", "date")
)
fig, ax = plt.subplots(figsize=(11, 3.8))
for k in TARGET_DIVISIONS:
    s = div_w.filter(pl.col("DIVISION") == k)
    ax.plot(s["date"], s["n"], color=DIV_COLOR[k], linewidth=1.2, label=f"Division {k}")
ax.set_ylabel("mean interventions / day (weekly)")
ax.set_title("Target series: Division 1 - 6, weekly mean of daily counts")
ax.legend(ncol=6, loc="upper left", bbox_to_anchor=(0, -0.12))
ax.set_ylim(0, None)
save(fig, "07_division_weekly")

fig, ax = plt.subplots(figsize=(8, 3.2))
data = [
    div.filter(pl.col("DIVISION") == k)["n"].drop_nulls().to_numpy()
    for k in TARGET_DIVISIONS
]
bp = ax.boxplot(
    data,
    tick_labels=[f"Div {k}" for k in TARGET_DIVISIONS],
    patch_artist=True,
    showfliers=True,
    flierprops={"markersize": 2, "markeredgecolor": MUTED},
    medianprops={"color": INK},
)
for patch, k in zip(bp["boxes"], TARGET_DIVISIONS, strict=True):
    patch.set_facecolor(DIV_COLOR[k])
    patch.set_edgecolor(DIV_COLOR[k])
ax.set_ylabel("interventions / day")
ax.set_title("Division-day distribution, 2020 - 2026 (long upper tail = storm days)")
save(fig, "07_division_box")

### 7f · Composition drift: which groups move the level?

Two documented breaks: first-responder (1-REPOND) calls almost vanish in spring 2020, and a drop
starts in late December 2025. Both are group-specific, not uniform.

In [ ]:
G = ["1-REPOND", "SANS FEU", "Alarmes-incendies", "AUTREFEU", "INCENDIE"]
monthly = (
    df.with_columns(
        g=pl.when(pl.col("DESCRIPTION_GROUPE").is_in(G))
        .then(pl.col("DESCRIPTION_GROUPE"))
        .otherwise(pl.lit("other / blank")),
        month=pl.col("date").dt.truncate("1mo"),
    )
    .group_by("month", "g")
    .agg(n=pl.len(), days=pl.col("date").n_unique())
    .with_columns(per_day=pl.col("n") / pl.col("days"))
    .pivot("g", index="month", values="per_day")
    .sort("month")
    .fill_null(0)
)
fig, ax = plt.subplots(figsize=(11, 3.6))
ax.stackplot(
    monthly["month"],
    *[monthly[g] for g in [*G, "other / blank"]],
    labels=[*G, "other / blank"],
    colors=SERIES[:6],
    edgecolor="#fcfcfb",
    linewidth=0.5,
)
ax.set_ylabel("interventions / day (monthly mean)")
ax.set_title("Composition by group: the breaks are first-responder breaks")
ax.legend(ncol=6, loc="upper left", bbox_to_anchor=(0, -0.12))
save(fig, "07_group_composition")

pr_day = daily_counts(df.filter(pl.col("DESCRIPTION_GROUPE") == "1-REPOND"), by=None)
covid_gap = pr_day.filter(pl.col("date").dt.year() == 2020, pl.col("n") < 5)["date"]
R["covid_pr_gap"] = {
    "first": str(covid_gap.min()),
    "last": str(covid_gap.max()),
    "days": covid_gap.len(),
}
print("2020 days with < 5 first-responder records:", R["covid_pr_gap"])

In [ ]:
same_window = pl.col("date").dt.ordinal_day() <= date(2026, 9, 24).timetuple().tm_yday
shift = (
    df.filter(
        pl.col("year").is_in([2025, 2026]),
        same_window,
        pl.col("DIVISION").is_in(TARGET_DIVISIONS),
    )
    .with_columns(pr=pl.col("DESCRIPTION_GROUPE") == "1-REPOND")
    .group_by("DIVISION", "pr", "year")
    .len()
    .pivot("year", index=["DIVISION", "pr"], values="len")
    .with_columns(pct_change=(100 * (pl.col("2026") / pl.col("2025") - 1)).round(1))
    .sort("DIVISION", "pr")
)
R["shift_2026_vs_2025"] = shift.to_dicts()
shift

**Reading.** Volume rose from ~227/day (2020, COVID) to ~358/day (2025), flat in 2023–24. The
2026 drop (Jan 1 – Sep 24 vs the same span of 2025) is concentrated in first-responder records
in Divisions 1–5, while non-first-responder volume is flat or up and Division 6 rises. Cause
unknown (a dispatch-protocol change is plausible but unverified). Implications: 2020 serves only
as lag history; any model anchored on last year's level will be biased in 2026; the baseline
window must be short enough to track level shifts.

### 7g · Seasonality (estimated on 2021 – 2024 only)

In [ ]:
train_div = div.filter(pl.col("date").is_between(*TRAIN)).drop_nulls()
dow = (
    train_div.group_by("DIVISION", pl.col("date").dt.weekday().alias("weekday"))
    .agg(n=pl.col("n").mean())
    .with_columns(index=pl.col("n") / pl.col("n").mean().over("DIVISION"))
    .sort("DIVISION", "weekday")
)
moy = (
    train_div.group_by("DIVISION", pl.col("date").dt.month().alias("month"))
    .agg(n=pl.col("n").mean())
    .with_columns(index=pl.col("n") / pl.col("n").mean().over("DIVISION"))
    .sort("DIVISION", "month")
)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.4), sharey=True)
for k in TARGET_DIVISIONS:
    s = dow.filter(pl.col("DIVISION") == k)
    axes[0].plot(
        s["weekday"],
        s["index"],
        marker="o",
        markersize=4,
        color=DIV_COLOR[k],
        label=f"Div {k}",
    )
    s = moy.filter(pl.col("DIVISION") == k)
    axes[1].plot(s["month"], s["index"], marker="o", markersize=4, color=DIV_COLOR[k])
axes[0].set_xticks(range(1, 8), ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"])
axes[1].set_xticks(range(1, 13), list("JFMAMJJASOND"))
axes[0].axhline(1, color=AXIS, lw=0.8)
axes[1].axhline(1, color=AXIS, lw=0.8)
axes[0].set_ylabel("index (1 = division mean)")
axes[0].set_title("Weekday profile: within ±9 %; Division 6 peaks on weekends")
axes[1].set_title("Month profile: summer high, storms distort Feb / Apr")
axes[0].legend(ncol=3, fontsize=8)
save(fig, "07_seasonality")
R["weekday_index_range"] = (
    dow.group_by("DIVISION")
    .agg(low=pl.col("index").min().round(3), high=pl.col("index").max().round(3))
    .sort("DIVISION")
    .to_dicts()
)
R["weekday_index_range"]

In [ ]:
city_train = city.filter(pl.col("date").is_between(*TRAIN))
y_city = city_train["n"].fill_null(strategy="forward").to_numpy().astype(float)
stl = STL(y_city, period=7, robust=True).fit()
strength_trend = max(0.0, 1 - np.var(stl.resid) / np.var(stl.trend + stl.resid))
strength_week = max(0.0, 1 - np.var(stl.resid) / np.var(stl.seasonal + stl.resid))
R["stl_citywide_2021_2024"] = {
    "trend_strength": round(strength_trend, 3),
    "weekly_strength": round(strength_week, 3),
}
print(R["stl_citywide_2021_2024"])

fig = stl.plot()
fig.set_size_inches(11, 6)
for ax in fig.axes:
    ax.xaxis.set_major_locator(mdates.YearLocator())
fig.axes[0].set_title("STL (period 7, robust), citywide 2021 - 2024", loc="left")
save(fig, "07_stl_citywide")

**Reading.** Weekly seasonality is real but small: the weekday index stays within ±9 % of each
division's mean, and STL's weekly-seasonal strength on the citywide series is only ~0.06 against
~0.33 for trend. Division 6 (Ville-Marie, Plateau) is the exception, with a weekend peak. Level
and irregular shocks dominate, so the part of the baseline that matters most is how quickly it
tracks the level, not the weekday alignment.

### 7h · Autocorrelation, stationarity and dispersion (2021 – 2024)

In [ ]:
rows = []
for k in TARGET_DIVISIONS:
    y = train_div.filter(pl.col("DIVISION") == k)["n"].to_numpy().astype(float)
    a = np.asarray(acf(y, nlags=28, fft=True), dtype=float)
    adf_p = float(adfuller(y, autolag="AIC", result_object=False)[1])
    # KPSS p-values come from a table capped at [0.01, 0.1]; 0.01 means "0.01 or less".
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", InterpolationWarning)
        kpss_p = float(kpss(y, regression="c", nlags="auto", result_object=False)[1])
    rows.append(
        {
            "division": k,
            "mean": round(float(y.mean()), 1),
            "var_over_mean": round(float(y.var(ddof=1) / y.mean()), 2),
            "acf_1": round(float(a[1]), 3),
            "acf_3": round(float(a[3]), 3),
            "acf_7": round(float(a[7]), 3),
            "acf_28": round(float(a[28]), 3),
            "adf_p": round(adf_p, 4),
            "kpss_p": round(kpss_p, 3),
        }
    )
dyn = pl.DataFrame(rows)
R["dynamics_2021_2024"] = dyn.to_dicts()
dyn

In [ ]:
k_big = int(dyn.sort("mean", descending=True)["division"][0])
y = train_div.filter(pl.col("DIVISION") == k_big)["n"].to_numpy().astype(float)
fig, axes = plt.subplots(1, 2, figsize=(11, 3))
plot_acf(y, lags=42, ax=axes[0], color=SERIES[0], vlines_kwargs={"colors": SERIES[0]})
plot_pacf(
    y,
    lags=42,
    ax=axes[1],
    color=SERIES[0],
    vlines_kwargs={"colors": SERIES[0]},
    method="ywm",
)
axes[0].set_title(f"ACF, Division {k_big}, 2021 - 2024")
axes[1].set_title(f"PACF, Division {k_big}, 2021 - 2024")
for ax in axes:
    ax.axvline(3, color=MUTED, lw=0.8, ls="--")
    ax.text(3.5, 0.85, "D-3 cutoff", fontsize=8, color=MUTED)
save(fig, "07_acf_pacf")

**Reading.** Every division is over-dispersed relative to Poisson (var/mean > 1), so a count
model would need a negative-binomial or quasi-Poisson form and prediction intervals must be
wider than √mean. Lag-1 autocorrelation is 0.36–0.59, but the first lag usable under the D−3
cutoff (lag 3) keeps only 0.16–0.38: most short-term persistence (storm aftermaths) is lost to
the publication lag. Division 6 is different: high autocorrelation even at lag 28 (a rising
level). ADF rejects a unit root and KPSS rejects level-stationarity in every division: the series
are mean-reverting around a slowly moving level, which favours recent-level features over a
fixed linear trend.

### 7i · Extreme days

In [ ]:
top_days = (
    df.group_by("date")
    .agg(
        n=pl.len(),
        vehicles=pl.col("NOMBRE_UNITES").sum(),
        pct_sans_feu=(100 * (pl.col("DESCRIPTION_GROUPE") == "SANS FEU").mean()).round(
            1
        ),
        top_type=pl.col("INCIDENT_TYPE_DESC").mode().first(),
    )
    .sort("n", descending=True)
    .head(12)
    .with_columns(weekday=pl.col("date").dt.strftime("%a"))
)
R["top_days"] = top_days.with_columns(pl.col("date").cast(pl.String)).to_dicts()
top_days

**Reading.** Surge days are weather days (ice storm, Debby rainfall, cold snaps): their top types
are electrical problems and flooding, and non-medical (SANS FEU) share jumps. No calendar
feature predicts them; public weather forecasts could, but they are outside the initial design.
For MAE evaluation these days dominate the error tail, so errors should also be reported with
and without the top 1 % of days.

## Step 8 · Privacy and transformations

- **Location**: moved to an intersection of the street segment (official methodology); the
  shared-point share above shows the snapping. Only one extraction is available locally, so
  re-drawing between extractions (reported by the team's 2020–2022 comparison) cannot be
  re-tested here. Irrelevant to division-level counts.
- **Coverage**: ~6 % of dispatch numbers are not published (step 2); reason undocumented.
- **Labels**: types may be revised after the initial report; the files show final labels only.
- **No personal data** is published; the analysis only uses counts.

## Step 9 · What the data do not measure

Response/arrival times, durations and clear times, responding units and their home stations,
crew size and staffing, unit availability or out-of-service periods, costs, call priority,
outcomes, and unpublished or unmet demand. Consequences: the target is **recorded volume, not
workload**; CASERNE/DIVISION are territories, not responders; no claims about staffing, vehicles,
coverage or response times.

## Step 10 · Fitness: can the target and baseline be built under the cutoff?

Preview of the v2 baseline family on the **validation year 2024** (history from 2020 – 2023; the
2025+ test period is not touched). Forecast for *D* = mean of the same weekday over the last *w*
weeks, all at or before *D−7* and hence inside the D−3 cutoff. Secondary reference: plain mean of
the 28 days ending *D−3*. This is a feasibility check, not the final window selection.

In [ ]:
WINDOWS = [4, 8, 13, 26, 52]
g = div.sort("DIVISION", "date")
preds = g.with_columns(
    *[
        pl.mean_horizontal(
            [pl.col("n").shift(7 * i).over("DIVISION") for i in range(1, w + 1)]
        ).alias(f"same_wd_{w}w")
        for w in WINDOWS
    ],
    plain_28d=pl.col("n").shift(3).rolling_mean(28, min_samples=20).over("DIVISION"),
)
val = preds.filter(pl.col("date").dt.year() == 2024).drop_nulls("n")
models = [f"same_wd_{w}w" for w in WINDOWS] + ["plain_28d"]
mae = (
    val.group_by("DIVISION")
    .agg(*[(pl.col(m) - pl.col("n")).abs().mean().round(2).alias(m) for m in models])
    .sort("DIVISION")
)
overall = val.select(
    *[(pl.col(m) - pl.col("n")).abs().mean().round(2).alias(m) for m in models]
).with_columns(DIVISION=pl.lit(0, dtype=mae["DIVISION"].dtype))
bias = val.select(*[(pl.col(m) - pl.col("n")).mean().round(2).alias(m) for m in models])
R["baseline_preview_2024"] = {
    "mae_all": overall.drop("DIVISION").to_dicts()[0],
    "bias_all": bias.to_dicts()[0],
    "mae_by_division": mae.to_dicts(),
}
print("bias (forecast - actual), all divisions:", bias.to_dicts()[0])
pl.concat([mae, overall.select(mae.columns)]).with_columns(
    pl.col("DIVISION").cast(pl.String).replace("0", "all")
)

**Fitness verdict.**

- **Target**: constructible. 14,748 division-days, median 53, no zeros, 2024-12-31 missing (not 0),
  DIVISION 0 excluded (600 rows).
- **Inputs at prediction time**: calendar and lagged counts through D−3; the observed 2-day
  publication lag supports this. Incident attributes of day D are excluded.
- **Unit**: DIVISION × day is dense and stable in code (1–6 throughout 2020 – 2026); CASERNE ×
  day is sparse (median 4, 9 % zeros) and does not nest in DIVISION.
- **Baseline preview (2024 validation)**: same-weekday windows of 13 and 26 weeks tie at MAE
  8.16 interventions/day (bias −0.4 to −0.7, i.e. slight under-forecast in a rising year); 4
  weeks is worst (8.89). The **plain 28-day mean ending D−3 is better than every same-weekday
  window (8.03)**: weekday alignment costs more in noise than it earns in signal.
- **Headroom**: modest. Weekday and month explain little; levels move and shocks are weather.
  A model can plausibly beat the same-weekday baseline by tracking the level faster, but the
  plain recent mean is the harder and more honest comparator; report it as the secondary
  reference the red-team review recommended.
- **Claims that exceed the evidence**: workload, staffing, vehicle needs, response times, the
  cause of the late-2025 drop, and "all SIM interventions" (published ≈ 94 %).

In [ ]:
Path("reports/eda_numbers.json").write_text(json.dumps(R, indent=2, default=str) + "\n")
print(f"wrote reports/eda_numbers.json with {len(R)} sections; figures in {FIG}/")